# Fase 2: Indicadores Técnicos Adaptativos, Momentum Cross-Sectional e Microestrutura de Mercado (Dollar Bars)

Este notebook demonstra:
1. **McGinley Dynamic**: Filtro adaptativo acelerado via Numba para eliminação de lag estrutural.
2. **Momentum Cross-Sectional**: Ranking percentil ajustado à volatilidade em horizontes múltiplos (1M, 3M, 12M).
3. **Amostragem por Dollar Bars**: Agrupamento baseado no fluxo de valor transacionado para estabilização de variância estatística.

In [1]:
# Cell 1: Importações e Simulação de Dados
import os
import sys
import pandas as pd
import numpy as np

# Adicionar raiz do repositório ao PATH
sys.path.insert(0, os.path.abspath('..'))

from src.features.technical import (
    compute_mcginley_dynamic,
    compute_cross_sectional_momentum,
    build_dollar_bars,
)

# Simulação de Série Temporal de Preços diários para 3 ativos
np.random.seed(42)
dates = pd.date_range(start="2024-01-01", periods=260, freq="B")
aapl_prices = 150 + np.cumsum(np.random.normal(0.1, 1.5, size=260))
msft_prices = 300 + np.cumsum(np.random.normal(0.2, 2.0, size=260))
googl_prices = 120 + np.cumsum(np.random.normal(-0.05, 1.2, size=260))

df_prices = pd.DataFrame({
    'AAPL': aapl_prices,
    'MSFT': msft_prices,
    'GOOGL': googl_prices
}, index=dates)

In [2]:
# Cell 2: Cálculo do McGinley Dynamic para AAPL
aapl_mcginley = compute_mcginley_dynamic(df_prices['AAPL'], period=14)
print("=== MCGINLEY DYNAMIC (ÚLTIMAS 5 OBSERVAÇÕES AAPL) ===")
print(pd.DataFrame({'Price': df_prices['AAPL'], 'McGinley_14': aapl_mcginley}).tail())

In [3]:
# Cell 3: Cálculo do Momentum Cross-Sectional
momentum_ranks = compute_cross_sectional_momentum(df_prices)

print("\n=== RANKING DE MOMENTUM CROSS-SECTIONAL (ÚLTIMA DATA) ===")
last_ranks = momentum_ranks.iloc[-1]
for ticker in ['AAPL', 'MSFT', 'GOOGL']:
    print(f"{ticker} - Rank 1M: {last_ranks[f'mom_rank_1M_{ticker}']:.2f} | Rank 12M: {last_ranks[f'mom_rank_12M_{ticker}']:.2f}")

In [4]:
# Cell 4: Simulação de Ticks e Construção de Dollar Bars
tick_dates = pd.date_range(start="2024-01-01 09:30", periods=1000, freq="15s")
df_ticks = pd.DataFrame({
    'timestamp': tick_dates,
    'price': 150 + np.random.normal(0, 0.2, 1000),
    'volume': np.random.randint(100, 1000, 1000)
})

# Threshold de 500.000 dólares por barra
dollar_bars = build_dollar_bars(df_ticks, threshold=500000.0)

print(f"\n=== AMOSTRAGEM DOLLAR BARS (Total Ticks: {len(df_ticks)} -> Total Barras: {len(dollar_bars)}) ===")
print(dollar_bars.head(3))